# 00 — Preparação da base

## Objetivo

Como transformar a base original em uma versão curta e legível para o curso?

Este notebook valida o contrato mínimo, traduz as colunas e salva uma cópia
preparada. O CSV bruto é somente lido.

In [ ]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import hashlib
import pandas as pd

from src.auxiliares import ALVO, COLUNA_ID, MAPEAMENTO_COLUNAS
from src.visual_utils import grafico_distribuicao_alvo

CAMINHO_BRUTO = RAIZ / "data" / "raw" / "UCI_Credit_Card.csv"
CAMINHO_PREPARADO = RAIZ / "data" / "processed" / "cartao_credito_portugues.csv"

## Como é a base?

A unidade observada é um registro por ID de cliente/conta. Não há datas por
linha: os meses aparecem apenas no significado das colunas.

In [ ]:
hash_bruto = hashlib.sha256(CAMINHO_BRUTO.read_bytes()).hexdigest()
dados_brutos = pd.read_csv(CAMINHO_BRUTO)

print(f"Dimensões: {dados_brutos.shape[0]:,} linhas x {dados_brutos.shape[1]} colunas")
print(f"SHA-256: {hash_bruto}")
dados_brutos.head()

## O contrato permite modelagem?

Antes de qualquer modelo, verificamos chave, granularidade, missing, duplicados
e outcome. A base não contém tratamento nem controle; este projeto é preditivo,
não causal.

In [ ]:
alvo_original = "default.payment.next.month"
resumo_qualidade = pd.Series({
    "linhas": len(dados_brutos),
    "colunas": dados_brutos.shape[1],
    "missing": int(dados_brutos.isna().sum().sum()),
    "duplicatas_exatas": int(dados_brutos.duplicated().sum()),
    "ids_duplicados": int(dados_brutos["ID"].duplicated().sum()),
    "taxa_inadimplencia": dados_brutos[alvo_original].mean(),
})
assert dados_brutos["ID"].is_unique
assert set(dados_brutos[alvo_original].unique()) == {0, 1}
resumo_qualidade.to_frame("resultado")

Os 30 mil IDs são únicos, não há valores ausentes e o target é binário.
Perfis coincidentes sem o ID são mantidos: eles não comprovam duplicação do
mesmo cliente.

## Como ficam os nomes em português?

Os valores e códigos permanecem iguais aos da fonte. Somente os nomes mudam.

In [ ]:
dicionario_colunas = MAPEAMENTO_COLUNAS.copy()
pd.DataFrame(
    dicionario_colunas.items(),
    columns=["nome_original", "nome_no_curso"],
)

In [ ]:
dados = dados_brutos.rename(columns=dicionario_colunas)
CAMINHO_PREPARADO.parent.mkdir(parents=True, exist_ok=True)
dados.to_csv(CAMINHO_PREPARADO, index=False)

assert dados.shape == dados_brutos.shape
assert COLUNA_ID in dados and ALVO in dados
print(f"Base preparada salva em: {CAMINHO_PREPARADO}")

## Qual é o desbalanceamento do problema?

A classe positiva é minoritária, mas ainda possui milhares de exemplos.

In [ ]:
distribuicao_alvo = (
    dados[ALVO].value_counts().sort_index()
    .rename_axis("inadimplente").to_frame("clientes")
)
distribuicao_alvo["proporcao"] = distribuicao_alvo["clientes"] / len(dados)

display(distribuicao_alvo)
fig = grafico_distribuicao_alvo(distribuicao_alvo)
fig.show()

## Existem códigos ou valores que exigem cuidado?

Os códigos não documentados são registrados, não corrigidos. Valores
monetários negativos podem representar saldo credor e também são preservados.

In [ ]:
colunas_categoricas = ["sexo", "escolaridade", "estado_civil"]
codigos = pd.Series({
    coluna: sorted(dados[coluna].unique().tolist())
    for coluna in colunas_categoricas
}, name="valores_observados")

faixas = dados[
    ["limite_credito", "idade", "valor_fatura_set", "valor_pago_set"]
].agg(["min", "median", "max"]).T
display(codigos.to_frame(), faixas)

## Resultado

A cópia em português preserva 30.000 linhas e 25 colunas. O ID será excluído
apenas das features; inadimplente será o outcome. A taxa positiva é 22,12%.